# eBay Car Sales Data
This project is a practice project to gain experience in the following:
- pandas
- data cleaning
- outlier detection
- string manipulation
- EDA

The instructions for the project were presented as follows:
1. Load the data and inspect column names and data types.
2. Clean column names and convert to snake_case.
3. Identify and handle outliers in price and mileage.
4. Explore the relationship between price and brand, mileage, and age.
5. Calculate mean prices by brand and identify which brands hold value best.
6. Summarize findings with visualizations.

## Cleaning Process

In [ ]:
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import pandas as pd

In [ ]:
df = pd.read_csv('autos.csv',encoding='latin1') # Use latin1 encoding because of the german alphabet.

print("Autos info: \n")
print(df.info())

Autos info: 

<class 'pandas.DataFrame'>
RangeIndex: 371528 entries, 0 to 371527
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   dateCrawled          371528 non-null  str  
 1   name                 371528 non-null  str  
 2   seller               371528 non-null  str  
 3   offerType            371528 non-null  str  
 4   price                371528 non-null  int64
 5   abtest               371528 non-null  str  
 6   vehicleType          333659 non-null  str  
 7   yearOfRegistration   371528 non-null  int64
 8   gearbox              351319 non-null  str  
 9   powerPS              371528 non-null  int64
 10  model                351044 non-null  str  
 11  kilometer            371528 non-null  int64
 12  monthOfRegistration  371528 non-null  int64
 13  fuelType             338142 non-null  str  
 14  brand                371528 non-null  str  
 15  notRepairedDamage    299468 non-null  str  
 16 

In [64]:
print("Head: \n")
print(df.head())

Head: 

           dateCrawled                            name  seller offerType  \
0  2016-03-24 11:52:17                      Golf_3_1.6  privat   Angebot   
1  2016-03-24 10:58:45            A5_Sportback_2.7_Tdi  privat   Angebot   
2  2016-03-14 12:52:21  Jeep_Grand_Cherokee_"Overland"  privat   Angebot   
3  2016-03-17 16:54:04              GOLF_4_1_4__3TÜRER  privat   Angebot   
4  2016-03-31 17:25:20  Skoda_Fabia_1.4_TDI_PD_Classic  privat   Angebot   

   price abtest vehicleType  yearOfRegistration    gearbox  powerPS  model  \
0    480   test         NaN                1993    manuell        0   golf   
1  18300   test       coupe                2011    manuell      190    NaN   
2   9800   test         suv                2004  automatik      163  grand   
3   1500   test  kleinwagen                2001    manuell       75   golf   
4   3600   test  kleinwagen                2008    manuell       69  fabia   

   kilometer  monthOfRegistration fuelType       brand notRepaired

Converting the languages from German to English allows for better readability for the english speaking audience.

In [65]:
df['seller'] = df['seller'].replace('privat','private')
df['seller'] = df['seller'].replace('gewerblich','commercial')
df['offerType'] = df['offerType'].replace('Angebot','offer')
df['offerType'] = df['offerType'].replace('Gesuch','request')

In [66]:
# Replace 'kleinwagen' with 'small car' and 'kombi' with 'station wagon'
df['vehicleType'] = df['vehicleType'].replace({'kleinwagen': 'small car', 'kombi': 'station wagon', 'andere': 'other'})

In [67]:
df['gearbox'] = df['gearbox'].replace({'manuell': 'manual', 'automatik': 'automatic'})

In [68]:
df.rename(columns={'powerPS': 'horsepower'}, inplace=True)

In [69]:
df['fuelType'] = df['fuelType'].replace({'benzin': 'petrol', 'elektro': 'electric', 'andere': 'other'})

In [70]:
df.rename(columns={'kilometer': 'mileage'}, inplace=True)

Many of the values include heavy outliers and potentially incorrect information in price. Values as high as $200,000,000,000 in the price column cannot realistically be used within the dataset to produce meaningful trends. We would have to decide whether the intention for the project is to produce insights on cheaper or more expensive cars. In this case, I opted to use cheaper cars as there were more values to collect trends from.

In [71]:
Q1 = df['price'].quantile(0.25)
Q3 = df['price'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['price'] >= Q1 - 1.5 * IQR) & (df['price'] <= Q3 + 1.5 * IQR)]

In [72]:
Q1 = df['yearOfRegistration'].quantile(0.25)
Q3 = df['yearOfRegistration'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['yearOfRegistration'] >= Q1 - 1.5 * IQR) & (df['yearOfRegistration'] <= Q3 + 1.5 * IQR)]

In [73]:
Q1 = df['horsepower'].quantile(0.25)
Q3 = df['horsepower'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['horsepower'] >= Q1 - 1.5 * IQR) & (df['horsepower'] <= Q3 + 1.5 * IQR)]

In [74]:
Q1 = df['mileage'].quantile(0.25)
Q3 = df['mileage'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['mileage'] >= Q1 - 1.5 * IQR) & (df['mileage'] <= Q3 + 1.5 * IQR)]

nrOfPictures would not have produced any meaningful insights as the pictures would not affect any potential trends hence it was removed.

In [75]:
df['notRepairedDamage'] = df['notRepairedDamage'].replace({'nein': 'no', 'ja': 'yes'})
df = df.drop(columns=["nrOfPictures"])

Creating a new cleaned CSV allows us to use the clean data to create visualisations without having to run the entire notebook between sessions.

In [76]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv("Cleaned Autos.csv", index=False)

## Visualisations

### Exploring the relationship between price and brand, mileage, and age

In [ ]:
clean_df = pd.read_csv('Cleaned Autos.csv')

clean_df['age'] = 2026 - clean_df['yearOfRegistration']

# Drop rows with missing values in 'mileage' or 'age'
valid = clean_df[['mileage', 'age']].dropna()

# Calculate Pearson correlation coefficient
corr, p_value = pearsonr(valid['mileage'], valid['age'])
print(f"Pearson correlation coefficient between mileage and age: {corr:.3f}")

# Calculate Spearman rank correlation coefficient
spearman_corr, spearman_p = spearmanr(valid['mileage'], valid['age'])
print(f"Spearman rank correlation between mileage and age: {spearman_corr:.3f}")

Pearson correlation coefficient between mileage and age: 0.161
Spearman rank correlation between mileage and age: 0.203


The Pearson correlation coefficient of 0.161 suggests that there is no relationship between mileage and age of the car. This is supported by the Spearman rank correlation of 0.203.

### Relationship between Price and Brand

In [ ]:
# Convert brand to categorical codes for correlation calculation
clean_df['brand_code'] = clean_df['brand'].astype('category').cat.codes

# Drop rows with missing values in 'price' or 'brand_code'
valid_spearman = clean_df[['price', 'brand_code']].dropna()

# Calculate Spearman rank correlation
spearman_corr, spearman_p = spearmanr(valid_spearman['price'], valid_spearman['brand_code'])
print(f"Spearman rank correlation between price and brand: {spearman_corr:.3f}")

# Calculate Pearson correlation between price and brand_code
pearson_corr, pearson_p = pearsonr(valid_spearman['price'], valid_spearman['brand_code'])
print(f"Pearson correlation coefficient between price and brand: {pearson_corr:.3f}")

Spearman rank correlation between price and brand: -0.111
Pearson correlation coefficient between price and brand: -0.125
